# Analysis

In [2]:
import pandas as pd

race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")
races = pd.read_csv("data/raw/f1db-races.csv")
quali_results = pd.read_csv("data/raw/f1db-races-qualifying-results.csv")
circuits = pd.read_csv("data/raw/f1db-circuits.csv")


races = races[races["year"].between(2010, 2026)]
race_results = race_results[race_results["year"].between(2010, 2026)]
quali_results = quali_results[quali_results["year"].between(2010, 2026)]

# One row per race
# Subset focuses only on the id -> race. Sum then adds up all the True values from duplicated() which only occur if an id is there for more than one time
print("Duplicate races:", races.duplicated(subset=["id"]).sum())

# One row per driver and race
print("Duplicate race results:", race_results.duplicated(subset=["raceId", "driverId"]).sum())

print("Duplicate qualifying results:", quali_results.duplicated(subset=["raceId", "driverId"]).sum())



Duplicate races: 0
Duplicate race results: 0
Duplicate qualifying results: 0


/var/folders/y6/yvr9h5p110qc_zbybx4w5p680000gn/T/ipykernel_16720/2600749808.py:3: DtypeWarning: Columns (31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  race_results = pd.read_csv("data/raw/f1db-races-race-results.csv")


- Races: Checks whether a race ID appears more than once.
- Race results: Checks whether a driver appears more than once in the same race.
- Qualifying results: Checks whether a driver appears more than once in the same qualifying session.

Especially important for new data updates

## Merge data sets

In [3]:
# how="inner" means that only for available ids of race_results, the values of race get joined (not too important here since the data is already very cleanly prepared)
# validate: raceId could be contained multiple times but Id must only appear once, else a MergeError would be triggered
# drop: drops id since we have raceId and id which are the same

model_data = race_results.merge(
    races[[
        "id",
        "date",
        "circuitId",
        "circuitType",
        "courseLength",
        "turns",
        "scheduledLaps"]],
    left_on="raceId",
    right_on="id",
    how="inner",
    validate="many_to_one"
).drop(columns="id")


# on: assigns to every driver the qualifying variables to the according race
# left: all the data of model_data stays, where the suiting parts from quali results get added
# rename is needed, since positionNumber is in the model_data used for the final position

model_data = model_data.merge(
    quali_results[[
        "raceId",
        "driverId",
        "q1Millis",
        "q2Millis",
        "q3Millis",
    ]],
    on=["raceId", "driverId"],
    how="left",
    validate="one_to_one"
)


## How to deal with missing values in gridPositionNumber

In [4]:
print(model_data["gridPositionText"].unique())

gridPositionNumber_nan = model_data[model_data["gridPositionText"].isna()]

['3' '2' '4' '1' '5' '7' '8' '6' '12' '11' '9' '10' '18' '13' '21' '15'
 '20' '14' 'PL' '19' '17' '16' '22' '24' '23' nan]


All the nan values are because of DNS or DNQ, thus they should be excluded. PL stands for "pit lane", meaning that the drivers had to start from the pit lane, which is more or less euqivalent as starting from the last position. Since the field hasn't always the same size, it can't be just set to 20.

In [5]:
model_data = model_data[model_data["gridPositionText"].notna()]

# transform("size"): size counts rows per group, where raceId is treated as the group

model_data["fieldSize"] = model_data.groupby("raceId")["driverId"].transform("size")


# .loc searches for the column gridPositionText with value "PL" and takes the related gridPositionNumber

model_data.loc[ model_data["gridPositionText"] == "PL", "gridPositionNumber"] = model_data["fieldSize"]

## Exlude and add features

In [6]:
columns_needed = [
    "raceId",
    "fieldSize",
    "year",
    "round",
    "date",
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType",
    "courseLength",
    "turns",
    "scheduledLaps",
    "gridPositionNumber",
    "gridPositionText",
    "qualificationPositionNumber",
    "q1Millis",
    "q2Millis",
    "q3Millis",
    "positionDisplayOrder",
    "positionText",
    "reasonRetired"
]


model_data = model_data[columns_needed].copy()

In [7]:
model_data["qualifying_fraction"] = (
    (model_data["qualificationPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["grid_fraction"] = (
    (model_data["gridPositionNumber"] - 1)
    / (model_data["fieldSize"] - 1)
)

model_data["finish_fraction"] = (
    (model_data["positionDisplayOrder"] - 1)
    / (model_data["fieldSize"] - 1)
)

Normalizing positions makes races with different field sizes comparable: finishing 20th out of 20 and 20th out of 22.

#### 5 race average

In [8]:
# transform: applies a computation on every group and returns its results back to the original row.
# shift(1): shifts all values one rows downwards, such that the result only gets included in the next races last five average
# rolling(5): uses five consecutive values
# min_periods says that the value should also be computed if only 4 instead of 5 values are available

model_data["driver_finish_5"] = (
    model_data.groupby("driverId")["finish_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)


#### 5 quali average

In [9]:
model_data["driver_quali_5"] = (
    model_data.groupby("driverId")["qualifying_fraction"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

#### Quali deviation feature

In [10]:
model_data["quali_vs_recent"] = (
    model_data["qualifying_fraction"]
    - model_data["driver_quali_5"]
)

#### Failure rate

In [11]:
model_data["failure"] = (model_data["positionText"].eq("DNF") | model_data["reasonRetired"].notna())

# .eq("DNF") is the same as == "DNF"
# .notna() gives back True, if a value isn't missing. (Everything that is missing in reasonRetiered hasn't retiered -> should return False)

model_data["failure_rate"] = (model_data.groupby("driverId")["failure"]
                                .transform(lambda failrate: failrate.shift(1).rolling(10, min_periods=5).mean()))

# mean values True and False as 0 and 1 leading to e.g. 0.4 -> 40%

#### Adding other average historical features

In [12]:
for window in [3, 5, 10, 20]:
    for source, name in [("finish_fraction", "finish"), ("qualifying_fraction", "quali"), ("failure", "failure_rate")]:
        model_data[f"driver_{name}_{window}"] = (model_data.groupby("driverId")[source]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))

This adds the average values not of the features computed above also for other race windows. i.e. 3, 5, 10 and 20 races

#### Standard deviations: consistency in races

In [13]:
for window in [5, 10]:
    model_data[f"driver_std_{window}"] = (
        model_data.groupby("driverId")["finish_fraction"].transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
    )

#### Several features for driver experience

Total experience in races

In [14]:
model_data["driver_experience"] = (
    model_data.groupby("driverId").cumcount()       # cumcount() counts the total apperances up to this point
)

Experience on this exact circuit (Possibly important if a new one gets intorduced)

In [15]:
model_data["driver_circuit_experience"] = (
    model_data.groupby(["driverId", "circuitId"]).cumcount()
)

How good a driver performs on a circuit. Once again only taking the previous 3 results into account, with a minimum of 1 race. If never raced this circuit before the drivers get a NaN, which later must be changed. Maybe to the median of the whold field or just ignored.

In [16]:
model_data["driver_circuit_form"] = (
    model_data.groupby(["driverId", "circuitId"])["finish_fraction"]
    .transform(lambda x:
                x.shift(1).rolling(3, min_periods=1).mean())
)

How often does the driver make a mistake/dnf?

In [17]:
for window in [3, 5, 10, 20]:
    model_data[f"driver_dnf_{window}"] = (
        model_data.groupby("driverId")["failure"]
        .transform(lambda x:
                    x.shift(1).rolling(window, min_periods=1).mean()))

#### Team Performance

We create a separate team table with one row per team per race, so rolling averages count races rather than individual drivers. Later these data sets get merged together

In [18]:
team_data = (
    model_data.groupby(
        ["raceId", "constructorId", "circuitId"],
        as_index=False                          # to ensure that the correct indizes remain, which is important for later merges
    )
    .agg(                                       # compresses each constructor per race into a row and creates the columns
        team_finish=("finish_fraction", "mean"),        # new column name = used variables, method
        team_quali=("qualifying_fraction", "mean"),
        team_dnf=("failure", "mean")
    )
)

Now done for different windows in history. Once again only using previous values (shift(1))

In [19]:
for window in [3, 5, 10, 20]:
    for source in ["team_finish", "team_quali", "team_dnf"]:
        team_data[f"{source}_{window}"] = (
            team_data.groupby("constructorId")[source]
            .transform(lambda x: 
                       x.shift(1).rolling(window, min_periods=1).mean()))

Now the average team form on a certain circuit is added to the team_data table

In [20]:
team_data["team_circuit_form"] = (
    team_data.groupby(["constructorId", "circuitId"])["team_finish"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

Finally we merge these features to the original data set.

In [21]:
# create a list via list comprehension with all the names of the team features
team_features = [f"{name}_{window}" for window in [3, 5, 10, 20] for name in ["team_finish", "team_quali", "team_dnf"]] + ["team_circuit_form"]


model_data = model_data.merge(
    team_data[["raceId", "constructorId"] + team_features],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one"
)

#### Differences

Between Driver and Teammate

In [22]:
model_data["driver_team_difference"] = (
    model_data["driver_finish_5"] - model_data["team_finish_5"]
)

Team Qualifying vs recent Team performance

In [23]:
model_data["team_quali_vs_recent"] = (
    model_data.groupby(["raceId", "constructorId"])["qualifying_fraction"].transform("mean") - model_data["team_quali_5"]
)

#### Qualifying Features

Gap to the leader in the session in %. If already droped out, a NaN will be created!

In [24]:
for stage in [1, 2, 3]:
    column = f"q{stage}Millis"

    best_time = model_data.groupby("raceId")[column].transform("min")       # min is here the minimum, not minute as one might think because of time feature

    model_data[f"q{stage}_gap"] = (model_data[column] / best_time - 1) * 100

#### Create a factor variable for street tracks

In [25]:
model_data["street"] = (
    model_data["circuitType"].eq("STREET").astype(int)          # checks if the street type is equal to STREET. If so it assigns the variable 1
)

Create different starting and qualifying variables for street tracks. This could be particularly important because street tracks are known for their narrowness, which makes it even more difficult to overtake. With those additional variables the model can account for that. If it isn't a street circuit the variable is equal to 0 which automatically eliminates it out of the model. Thus, it works like a weighted factor variable.

In [26]:
model_data["grid_street"] = (
    model_data["grid_fraction"] * model_data["street"]
)

model_data["qualifying_street"] = (
    model_data["qualifying_fraction"] * model_data["street"]
)

#### Pit-Lane Start
The model should be able to give a start from the pit lane a different weight.

In [27]:
model_data["pit_lane"] = model_data["gridPositionText"].eq("PL").astype(int)

#### Failure Rates on circuit
Again we create a new data set to use rolling for the last 5 races, not driver performances

In [28]:
circuit_data = (
    model_data.groupby(["raceId", "circuitId"], as_index=False)
    .agg(circuit_dnf=("failure", "mean"))                           # create the variable circuit_dnf which is the mean failure rate from the failure feature
)


# use the last 5 races on this circuit to create the rolling average
circuit_data["circuit_dnf_history"] = (
    circuit_data.groupby("circuitId")["circuit_dnf"]
    .transform(lambda x:
                x.shift(1).rolling(5, min_periods=1).mean())
)


# merge the data into the original data set
model_data = model_data.merge(
    circuit_data[["raceId", "circuit_dnf_history"]],
    on="raceId",
    how="left",
    validate="many_to_one"
)

## Model feature selection

#### Categorical features

In [29]:
categorical_features = [
    "driverId",
    "constructorId",
    "circuitId",
    "circuitType"
]

#### Numerical features

In [30]:
numeric_features = [
    "gridPositionNumber",
    "qualificationPositionNumber",
    "grid_fraction",
    "qualifying_fraction",
    "pit_lane",
    "street",
    "grid_street",
    "qualifying_street",
    "courseLength",
    "turns",
    "scheduledLaps",
    "q1_gap",
    # "q2_gap",
    # "q3_gap",
    "driver_std_5",
    "driver_std_10",
    "driver_experience",
    "driver_circuit_form",
    "driver_circuit_experience",
    "team_circuit_form",
    "circuit_dnf_history",
    "driver_team_difference",
    "quali_vs_recent",
    "team_quali_vs_recent"
]

rolling_window_features = ["driver_finish","driver_quali","driver_dnf","team_finish","team_quali","team_dnf"]

for window in [3, 5, 10, 20]:
    for name in rolling_window_features:
        numeric_features.append(f"{name}_{window}")

q2_gap and q3_gap are left out because their missing values often indicate elimination in an earlier qualifying stage, making median imputation potentially misleading.

## Split Data for CV and Final Test

In [31]:
# This data is used to find the best model via CV
development_data = model_data[model_data["year"].between(2010, 2024)].copy()


# This data is for the final test
test_data = model_data[model_data["year"] == 2025].copy()

#### CV rolling window for 5 years

In [32]:
cv_splits = []

for validation_year in [2022, 2023, 2024]:
    train = development_data[development_data["year"].between(validation_year - 5, validation_year - 1)].copy()

    valid = development_data[development_data["year"] == validation_year].copy()

    cv_splits.append({
        "validation_year": validation_year,
        "train": train,
        "valid": valid})

## Early Model Brainstorm
1. Local polynomial regression: Could flexibly model a few relationships, such as starting versus finishing position, but becomes less practical with many features.

2. Random Forest: Could capture nonlinear patterns and interactions without specifying them manually.
3. Gradient boosting / XGBoost: Could combine many small decision trees to capture subtle patterns in structured race data.
4. Ridge regression: Could suit our many correlated features, such as qualifying position and recent form. Regularization may reduce overfitting.

5. Mixed-effects models: Could account for repeated observations of the same drivers, teams, and circuits through group-specific effects.

## Model Processing 

We use a preprocessing pipeline to fill missing values, standardize numerical features, and encode categorical features.


Fill missing values: Some information is unavailable for example when a driver has never raced at a circuit before. Since some machine learing models cannot process missing values, we replace missing numbers with the median of that feature in the training data. We also add an indicator so the model can distinguish missing information from an actual recorded value.


Standardize numerical features: Features use different scales, such as circuit length in kilometres and failure rates between 0 and 1. We rescale them using their training mean and standard deviation so machine learnign models penaltizes  features comparably, regardless of their units. Especially if we use a ridge model this would be particularilly important, else this could be ignored.


Encode categorical features: MLs needs numbers rather than labels such as driver or team names. We create separate 0/1 columns for each category, indicating which driver, team, or circuit a row belongs to without implying an artificial numerical ranking.

For example, instead of assigning Ferrari = 1, McLaren = 2, and Mercedes = 3—which could create 


Team	    Ferrari	McLaren	Mercedes


Ferrari	    1	    0	    0


McLaren	    0	    1	    0


Mercedes	0	    0	    1


This identifies the team without implying that one team is numerically “greater” than another.


These steps are fitted only on the training data in each cross-validation fold and then applied to the validation data, preventing data leakage.

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# SimpleImputer is a class that estimates NaN values. Here they get replaced with the median value
# StandardScaler creates an object to standardize as explained above. It processes the data after the imputer: subtracts the mean and devides with the std. deviation
# Pipeline is used to execute step after step. Each step is a tuple with its name and what its method
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                             ("scaler", StandardScaler())])


# OneHotEncoder creates the described factor variables to distinguish things like constructors with numbers not with names
# handle_unknown decides how to deal with e.g. drivers who never were in the training data. All the created collumns are thus set to 0 for this e.g. driver
categorical_pipeline = Pipeline([ #("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),       probably not necessary because our data is pretty clean
                                 ("encoder", OneHotEncoder(handle_unknown="ignore"))])


# The preprocessor defines how each feature will be prepared once data is passed in: numerical features are filled and scaled, while categorical features are converted into 0/1 columns.
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                                  ("categorical", categorical_pipeline, categorical_features)])

## 

## Model Training and Validation 

#### Linear Regression

In [57]:
from sklearn.linear_model import LinearRegression
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error


# Creates the pipeline for preprocessing the date in the first step with the model processig step. 
# Afterwards in the step regression a linear regression is fitted
model = Pipeline([("preprocessor", preprocessor), ("regression", LinearRegression())])



# create an empty list to later store the MAE of each iteration
cv_results = []


# Take the cv_splits list from the section above and create for every split a train and validation split

# split 1: 2017 - 21 train, 2022 valid
# split 2: 2018 - 22 train, 2023 valid
# split 3: 2019 - 23 train, 2024 valid
for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])


    # each fold trains its own model. Clone clones the whole pipeline from model, s.t. no model learns from the others
    fold_model = clone(model)

    # here we fit the model. this is done by giving two parameters. firstly all the numeric and categorical features
    # then declaring which variable is the depending one.
    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])


    # here the prediction is made. this is done by only providing the features of the validation set. 
    # with our previous setted settings the predicted values should be the finish_fraction. 
    # Obviously the smaller the value, the better the predicted placement.
    predictions = fold_model.predict(valid[numeric_features + categorical_features])


    # After the last step the predictions are some arbitrary values without a concrete predicited place. 
    # Thus, now these "score values" get ranked. As stated above, the smaller the value, the better the predicted placement.
    # Therefore, we will rank in an acsending manner: smalles score -> place no. 1, etc.

    # pd.Series is exactly used for creating such series. predictions are the values that need to be ranked by the series.
    # index = valid.index ensures that each predction is assigned to its original driver
    # groupby(valid["raceId"]) makes sure that each raced is analyzed on its own
    # rank(method="first") then finally assigns the places within a race form 1 ascending
    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")


    # this constructs our statistic to compare the quality of our model.
    #  The mean absolute error is computed by ∑|true placement - predicted placement|/(dirvers * races)
    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)


    # append the result to the list, for each year creating a dict
    cv_results.append({"validation_year": split["validation_year"],"mae": mae})


# creates a pandas data set for the cv_results
cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,4.392694
1,2023,3.725400
2,2024,3.164927


Average validation MAE: 3.761


In [35]:
# store the result so that it won't get overwritten by the next model
linear_results = cv_results_df.copy()

#### Random Forest

In [56]:
from sklearn.ensemble import RandomForestRegressor

# a new model pipeline is declared: preprocessing still is the same process of replacing NaNs etc.
# but now the regression is done by a random forest.
# n_estimators: Trains n decision trees and averages over their predictions
# min_sample_leaf: Each finalized group of a tree must contain at leat n observations. Here mae is optimized with n = 31
# ranodm_state: same as set seed, used to make results reproducable
# n_jobs: used to declare which cpu cores get used (idk)
model = Pipeline([("preprocessor", preprocessor),
    ("regression", RandomForestRegressor(n_estimators=300, min_samples_leaf=31, random_state=13, n_jobs=-1))])

# same as in the linear regression
cv_results = []

for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])

    fold_model = clone(model)

    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])

    predictions = fold_model.predict(valid[numeric_features + categorical_features])

    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")

    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)

    cv_results.append({"validation_year": split["validation_year"],"mae": mae})

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,3.538813
1,2023,3.331808
2,2024,2.931106


Average validation MAE: 3.267


In [37]:
forest_results = cv_results_df.copy()

#### xgboost

The trees get trained one after the other, trying to correct the mistakes the trees before have done.

In [55]:
from xgboost import XGBRegressor

# n_estimators: n trees
# learing_rate: each tree only corrects a small part to make the learning process more carefull
# max_depth: each tree can at most do n splits
# after each split, each group must at least contain n observations
# subsample: each tree can only use this fraction of the training set. the partition is random
# colsample_bytree: each tree can only use this fraction of features
# reg_lambda: Higher values penalize large leaf weights more strongly, especially in small leaves.
# objective: declares how to measure errors and how to minimize them
# tree_method: optimizes speed by apporximation via sepperation in bins. "auto" is the default value
model = Pipeline([("preprocessor", preprocessor),
    ("regression", XGBRegressor(n_estimators=300, learning_rate=0.02, max_depth=20, 
                                min_child_weight=100, subsample=0.6, colsample_bytree=0.6, 
                                reg_lambda=9, objective="reg:absoluteerror", tree_method="auto", 
                                random_state=0, n_jobs=2))])

# try different approaches for the objective e.g. absoluteerror and squarederror
# at first glance abs error seems to perform better

cv_results = []

for split in cv_splits:
    train = split["train"]
    valid = split["valid"].sort_values(["raceId", "driverId"])

    fold_model = clone(model)

    fold_model.fit(train[numeric_features + categorical_features], train["finish_fraction"])

    predictions = fold_model.predict(valid[numeric_features + categorical_features])

    predicted_positions = pd.Series(predictions, index=valid.index).groupby(valid["raceId"]).rank(method="first")

    mae = mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)

    cv_results.append({"validation_year": split["validation_year"],"mae": mae})

cv_results_df = pd.DataFrame(cv_results)

display(cv_results_df)

print(f"Average validation MAE: {cv_results_df["mae"].mean():.3f}")

,validation_year,mae
0,2022,3.525114
1,2023,3.382151
2,2024,2.843424


Average validation MAE: 3.250


In [39]:
xgboost_results = cv_results_df.copy()

##### Searching the best configutarion of parameter settings

Optuna uses Bayesian optimization to search for suitable model parameters. It uses previous trial results to suggest promising configurations, trying to minimize the average CV mae across multiple years and random seeds.

In [47]:
import optuna
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# both feature groups get unified
# each parameter configuration is tested on 3 seeds: 0, 13, 42
features = numeric_features + categorical_features
seeds = [0, 13, 67]


# Optuna uses this function to evaluate each parameter configuration
# a trial represents one optimization attempt and suggests the parameters to test
def objective(trial):

    # Defines the range of values Optuna can search through for each parameter
    parameters = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1200, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 5),
        "min_child_weight": trial.suggest_float("min_child_weight", 5, 200, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0), 
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0), 
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 100, log=True)
        }

    # list to store the 3 maes per iteration
    seed_maes = []

    # evaluates for each seed
    for seed in seeds:

        # yearly is not really an accurate name but I had no better idea: It stores the mae of each spilt
        yearly_maes = []
        # takes the defined data splits
        for split in cv_splits:
            train = split["train"]

            # this is an incredible important addition, to make sure, that we don't use information of the races based on the order of the drivers in the data set. 
            # This makes sure that the drivers are sorted in a non-biased way, e.g. ther driverId/name
            valid = split["valid"].sort_values(["raceId", "driverId"])

            # create a new Pipeline for each candidate, which represents some "arbitrary" setting of the parametes
            # here we need to clone the preprocessor, which happens above in the clone(model), 
            # to ensure he computes every time the new medians, etc. for each split
            # **parameters: ** distributes the parameters to all requiered spots. sort of unpacks it in the function call
            # the parametes that weren't already declared in the "parameters"
            candidate = Pipeline([
                ("preprocessor", clone(preprocessor)),
                ("regression", XGBRegressor(**parameters, objective="reg:absoluteerror", 
                                            tree_method="auto", random_state=seed, n_jobs=2))])

            # just as in the upper models values are fitted/ model gets trained with the train sets
            candidate.fit(train[features], train["finish_fraction"])

            # and predictions are made
            predictions = pd.Series(candidate.predict(valid[features]), index=valid.index)

            predicted_positions = predictions.groupby(valid["raceId"]).rank(method="first")

            # for each split the mae is appended
            yearly_maes.append(mean_absolute_error(valid["positionDisplayOrder"], predicted_positions))

        # the average for each seed is created and appended to the list
        seed_maes.append(sum(yearly_maes) / len(yearly_maes))

    # store additional information in each trial of optuna
    # firstly the seed maes and additionaly the standard deviation

    # this can be heloful, if we later want to call for e.g. the best trials attributes
    trial.set_user_attr("seed_maes", seed_maes)
    trial.set_user_attr("seed_std", float(pd.Series(seed_maes).std()))


    # returns the value optuna should optimize: the average over all seeds, which
    return sum(seed_maes) / len(seed_maes)

# direction = minimize declares, that the return value abouve should get minimized -> find the smallest MAE possible
# sampler=TPESamper: seed: "controlls" the ranomization,
# n_startup_trials: defines how many completly random settings are made to explore the data set
# afterwards the sampler parts the findings in a group with good mae values
# later it estimates which parameters are how distributed in which groups ~~~
# at last it recommends futher vaues which should be more likely
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123, n_startup_trials=30))

# Uses the existing configuration as the starting reference
study.enqueue_trial({'n_estimators': 100, 'learning_rate': 0.005318421512175779, 
                     'max_depth': 1, 'min_child_weight': 6.607289473852337, 
                     'subsample': 0.9850072285777673, 'colsample_bytree': 0.7925494017883324, 
                     'reg_lambda': 0.11796923067693268})


# calls our defined optimize function and repeats it 150 times
study.optimize(objective, n_trials=150, n_jobs=1)

print("Best CV MAE:", round(study.best_value, 3))
print("Best parameters:", study.best_params)
print("MAE per seed:", study.best_trial.user_attrs["seed_maes"])
print("Seed standard deviation:", study.best_trial.user_attrs["seed_std"])

# shows the best 10 configurations
display(study.trials_dataframe().sort_values("value").head(10))

[I 2026-11-03 13:59:15,534] A new study created in memory with name: no-name-27a64044-90a6-4fb1-a0d6-533add97f6b2
[I 2026-11-03 13:59:15,937] Trial 0 finished with value: 3.596783424192189 and parameters: {'n_estimators': 100, 'learning_rate': 0.005318421512175779, 'max_depth': 1, 'min_child_weight': 6.607289473852337, 'subsample': 0.9850072285777673, 'colsample_bytree': 0.7925494017883324, 'reg_lambda': 0.11796923067693268}. Best is trial 0 with value: 3.596783424192189.
[I 2026-11-03 13:59:19,022] Trial 1 finished with value: 3.266324303258113 and parameters: {'n_estimators': 900, 'learning_rate': 0.013232213446803209, 'max_depth': 2, 'min_child_weight': 38.21284198951867, 'subsample': 0.8597344848927815, 'colsample_bytree': 0.7115532300622305, 'reg_lambda': 87.557347250565}. Best is trial 1 with value: 3.266324303258113.
[I 2026-11-03 13:59:21,765] Trial 2 finished with value: 3.3008821415296716 and parameters: {'n_estimators': 850, 'learning_rate': 0.025666386987120496, 'max_depth'

Best CV MAE: 3.206
Best parameters: {'n_estimators': 250, 'learning_rate': 0.018813189427787296, 'max_depth': 4, 'min_child_weight': 35.91903387824371, 'subsample': 0.5415489165528182, 'colsample_bytree': 0.545429682036317, 'reg_lambda': 1.0388655930625168}
MAE per seed: [3.203445199585987, 3.2064963056119216, 3.2095474116378564]
Seed standard deviation: 0.0030511060259346134


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_lambda,params_subsample,user_attrs_seed_maes,user_attrs_seed_std,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
136,136,3.206496,2026-11-03 14:04:48.404684,2026-11-03 14:04:50.194242,0 days 00:00:01.789558,0.545430,0.018813,4,35.919034,250,1.038866,0.541549,"[3.203445199585987, 3.2064963056119216, 3.2095...",0.003051,NaN,"{""colsample_bytree"": 0.545429682036317, ""learn...",COMPLETE
94,94,3.210947,2026-11-03 14:02:53.994209,2026-11-03 14:02:55.845124,0 days 00:00:01.850915,0.524929,0.018045,4,51.958495,300,1.497578,0.551622,"[3.2155054101316742, 3.1933188017431946, 3.224...",0.015849,NaN,"{""colsample_bytree"": 0.5249288706409675, ""lear...",COMPLETE
140,140,3.213474,2026-11-03 14:04:57.150423,2026-11-03 14:04:59.104985,0 days 00:00:01.954562,0.566586,0.016888,4,39.693677,300,1.474030,0.557931,"[3.2188135962939914, 3.2163691063160775, 3.205...",0.007236,NaN,"{""colsample_bytree"": 0.5665858521431353, ""lear...",COMPLETE
114,114,3.215061,2026-11-03 14:03:48.403174,2026-11-03 14:03:52.868252,0 days 00:00:04.465078,0.533363,0.019721,4,43.611489,300,1.171548,0.549301,"[3.219138751221403, 3.210791503530215, 3.21525...",0.004177,NaN,"{""colsample_bytree"": 0.5333629680950299, ""lear...",COMPLETE
87,87,3.215712,2026-11-03 14:02:39.064285,2026-11-03 14:02:41.069857,0 days 00:00:02.005572,0.521518,0.017023,4,51.039851,350,0.965151,0.563751,"[3.215118048428226, 3.2161015771864547, 3.2159...",0.000523,NaN,"{""colsample_bytree"": 0.521518079379609, ""learn...",COMPLETE
33,33,3.217006,2026-11-03 14:00:44.907560,2026-11-03 14:00:47.455127,0 days 00:00:02.547567,0.573596,0.011376,4,32.991403,450,1.222220,0.561579,"[3.212179808980624, 3.235909385370803, 3.20292...",0.017012,NaN,"{""colsample_bytree"": 0.5735961678387194, ""lear...",COMPLETE
62,62,3.217081,2026-11-03 14:01:46.134449,2026-11-03 14:01:48.091670,0 days 00:00:01.957221,0.567037,0.017735,4,40.632083,350,0.494158,0.569090,"[3.2069621910996378, 3.2210138401460076, 3.223...",0.008835,NaN,"{""colsample_bytree"": 0.5670371352248309, ""lear...",COMPLETE
108,108,3.217229,2026-11-03 14:03:32.788729,2026-11-03 14:03:34.965393,0 days 00:00:02.176664,0.515437,0.017148,4,43.064984,350,0.596182,0.538385,"[3.210534423393831, 3.206605689192507, 3.23454...",0.015125,NaN,"{""colsample_bytree"": 0.5154372058077241, ""lear...",COMPLETE
92,92,3.217595,2026-11-03 14:02:50.705038,2026-11-03 14:02:52.254091,0 days 00:00:01.549053,0.529029,0.016960,4,46.284659,250,1.057351,0.540450,"[3.2216111051812892, 3.2174287738413287, 3.213...",0.003936,NaN,"{""colsample_bytree"": 0.5290286728352777, ""lear...",COMPLETE
142,142,3.218325,2026-11-03 14:05:01.254843,2026-11-03 14:05:03.230507,0 days 00:00:01.975664,0.509792,0.016445,4,64.314731,350,0.780491,0.542410,"[3.2099452223345435, 3.211477741343004, 3.2335...",0.013209,NaN,"{""colsample_bytree"": 0.5097922356046107, ""lear...",COMPLETE


The baseline predicts that drivers will finish in their starting order. It provides a simple benchmark to check whether our machine learning models improve predictions beyond the information already contained in the starting grid.

In [48]:
baseline_results = []

for split in cv_splits:
    valid = split["valid"].sort_values(["raceId", "driverId"])

    # here we profit from our earlier declaration of how to trat missing values in grid position.
    # now we don't need to do any kind of filtering etc.
    scores = valid["gridPositionNumber"]
    
    predicted_positions = scores.groupby(valid["raceId"]).rank(method="first")

    baseline_results.append({
        "validation_year": split["validation_year"], 
        "mae": mean_absolute_error(valid["positionDisplayOrder"], predicted_positions)})

baseline_results_df = pd.DataFrame(baseline_results)

display(baseline_results_df)

print("Average baseline MAE:",round(baseline_results_df["mae"].mean(), 3))

,validation_year,mae
0,2022,3.954338
1,2023,3.729977
2,2024,2.893528


Average baseline MAE: 3.526


## 2025 Evaluation

In [65]:
train_final = model_data[model_data["year"].between(2020, 2024)].copy()

# splited in the split for CV section
test_final = test_data.sort_values(["raceId", "driverId"])

features = numeric_features + categorical_features

# takes the best parameters from the optuna study
best_parameters = study.best_params.copy()


test_results = []
# prepares a list to store the later predictions
test_predictions = test_final[["raceId", "driverId", "positionDisplayOrder"]].copy()

for seed in [0, 13, 123]:

    # again prepare for each seed a new clean pipeline
    final_model = Pipeline([
        ("preprocessor", clone(preprocessor)),

        # unpack the best parameters into the regresison function and a add the missing parameters
        ("regression", XGBRegressor(**best_parameters, objective="reg:absoluteerror", 
                                    tree_method="auto", random_state=seed, n_jobs=2))])

    # fit the model
    final_model.fit(train_final[features], train_final["finish_fraction"])


    # predict the positions scores
    predictions = pd.Series(final_model.predict(test_final[features]), index=test_final.index)

    # rank the scores ascending
    positions = predictions.groupby(test_final["raceId"]).rank(method="first")

    # store the predictions in the according seed column for potential further investigation
    test_predictions[f"prediction_seed_{seed}"] = positions

    # store the mean average error in the result list
    test_results.append({"seed": seed, "mae": mean_absolute_error(test_final["positionDisplayOrder"], positions)})

# create the baseline model on the same 2025 observations
baseline_scores = (test_final["gridPositionNumber"])

baseline_positions = baseline_scores.groupby(test_final["raceId"]).rank(method="first")

baseline_mae = mean_absolute_error(test_final["positionDisplayOrder"], baseline_positions)

# create an other baseline (which some studies argue is the better one) to check our model with

quali_scores = test_final["qualificationPositionNumber"].fillna(float("inf"))       # float("inf") makes sure that drivers with missing values get appended at the end of the grid
quali_positions = quali_scores.groupby(test_final["raceId"]).rank(method="first")   # creates the prediction

quali_mae = mean_absolute_error(test_final["positionDisplayOrder"], quali_positions)


test_results_df = pd.DataFrame(test_results)
display(test_results_df)

print("Average XGBoost MAE:", round(test_results_df["mae"].mean(), 3))
print("Seed standard deviation:", round(test_results_df["mae"].std(), 3))
print("Startposition baseline MAE:", round(baseline_mae, 3))
print("Qualifying baseline MAE:", round(quali_mae, 3))

,seed,mae
0,0,3.303983
1,13,3.291405
2,123,3.287212


Average XGBoost MAE: 3.294
Seed standard deviation: 0.009
Startposition baseline MAE: 3.333
Qualifying baseline MAE: 3.312


## Walk-forward Evaluation

Now as we roughly know which model type seems to be promising, we try to predict the 2025 races with ongoing generation of new models, to simulate a "real" application of our future model. This is called walk-forward evaluation, since for every race weekend we estimate a new model. For efficiency reasons for now we leave the hyperparameter as they are.

In [67]:
walk_data = model_data.copy()           # .copy() to leave the original data as it is
walk_data["date"] = pd.to_datetime(walk_data["date"])   # pd.to_datetime converts the dates in the column to real date value. This is especially important to set date bound

features = numeric_features + categorical_features


def walk_forward(model, year):
    walk_results = []

    # .eq marks the wanted rows, .loc then selects them
    evaluation_data = walk_data.loc[walk_data["year"].eq(year)]

    # Iterating over groupby returns a tuple for each race:
    # (race ID, data frame with all driver rows and columns for that race)
    for race_id, race in evaluation_data.groupby("raceId"):

        race_date = race["date"].iloc[0]                   # for every race it takes the date form the first row, since all the dates are the same, since they stem form the same raceId
        window_start = race_date - pd.DateOffset(years=5)  # for each race, the training data window is set to all races from 5 years ago

        # .ge(): on or after the given date
        # .lt(): strictly before the given date
        # The training window starts five years before the race and excludes the race being predicted.
        train = walk_data.loc[walk_data["date"].ge(window_start) & walk_data["date"].lt(race_date)]

        # Use a tie-break that does not depend on the actual finishing order!
        # This is important since the drivers are sorted in the data acoording to their actual finishing position.
        # To prevent that we take any information, we must sort the drivers within that race in an unbiased order, e.g. their driverId / names
        race = race.sort_values("driverId")

        # Clone the entire pipeline to start with an unfitted model for each race.
        race_model = clone(model)

        # finish_fraction ~ features
        race_model.fit(train[features], train["finish_fraction"])

        scores = pd.Series(race_model.predict(race[features]), index=race.index)    # assigns each driver its predicted score
        positions = scores.rank(method="first")         # ranks the drivers according to their score

        # Creates a data frame with predictions and errors for this race
        walk_results.append(pd.DataFrame({
            "raceId": race_id, "driverId": race["driverId"], "positionDisplayOrder": race["positionDisplayOrder"],
            "predicted_position": positions, "absolute_error": (race["positionDisplayOrder"] - positions).abs()}))

    # UPDATED: Combines the individual race tables for this model and year.
    return pd.concat(walk_results, ignore_index=True)

#### xgboost & ridge

In [75]:
from sklearn.linear_model import Ridge

# create a new pipeline which uses once again the cloned, i.e. clean preprocessor and now as regression model the ridge model
# alpha: hyperparameter to punish large coefficients, the larger the harder the punishment
ridge_pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=300))])

validation_results = []

for year in [2022, 2023, 2024]:
    # use the walk_forward function from above to predict the results with ridge_pipeline as model
    ridge_predictions = walk_forward(ridge_pipeline, year=year)

    # ["absolute_error"].mean(): "absolute_error" is created automatically in our walk_forward function
    validation_results.append({"model": "Ridge", "year": year, "mae": ridge_predictions["absolute_error"].mean()})

    xgboost_seed_maes = []

    # for every seed we fit and predict the results with the hyperparameter from the optuna study
    for seed in [0, 13, 123]:
        xgboost_pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("regression", XGBRegressor(**best_parameters, objective="reg:absoluteerror",
                tree_method="auto", random_state=seed, n_jobs=2))])

        # also for this model we use our own function to predict the results but with the xgboost_pipeline
        xgboost_predictions = walk_forward(xgboost_pipeline, year=year)

        xgboost_seed_maes.append(xgboost_predictions["absolute_error"].mean())

    validation_results.append({"model": "XGBoost", "year": year, "mae": sum(xgboost_seed_maes) / len(xgboost_seed_maes)})

comparison = pd.DataFrame(validation_results).pivot(index="model", columns="year", values="mae")

comparison["Average MAE"] = comparison.mean(axis=1)

display(comparison.sort_values("Average MAE").round(3))

year,2022,2023,2024,Average MAE
model,,,,
Ridge,3.553,3.190,2.802,3.181
XGBoost,3.463,3.259,2.830,3.184


Optimize the alpha value for the ridge model:

In [ ]:
ridge_alpha_results = []

for alpha in [776, 777, 778]:
    candidate = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=alpha))])

    result = {"alpha": alpha}

    for year in [2022, 2023, 2024]:
        predictions = walk_forward(candidate, year=year)
        result[year] = predictions["absolute_error"].mean()

    result["Average MAE"] = (result[2022] + result[2023] + result[2024]) / 3
    ridge_alpha_results.append(result)

ridge_alpha_comparison = pd.DataFrame(ridge_alpha_results).sort_values("Average MAE")

display(ridge_alpha_comparison.round(3))

best_alpha = float(ridge_alpha_comparison.iloc[0]["alpha"])     # use the firts line, since its sorted for average MAE and take the alpha value
print(f"Best tested alpha: {best_alpha:g}")

,alpha,2022,2023,2024,Average MAE
1,777,3.557,3.208,2.764,3.176
2,778,3.557,3.208,2.764,3.176
0,776,3.557,3.208,2.768,3.178


Best tested alpha: 777


#### 2025 Test

In [94]:
# create a pipeline for the ridge model with its best alpha value
ridge_pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("regression", Ridge(alpha=best_alpha))])


# create a new prediction with our function
ridge_predictions_2025 = walk_forward(ridge_pipeline, year=2025)

# and calculate its MAE
ridge_mae_2025 = ridge_predictions_2025["absolute_error"].mean()

xgboost_seed_maes_2025 = []

# for every seed we fit and predict the results with the hyperparameter from the optuna study
for seed in [0, 13, 123]:
    xgboost_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", XGBRegressor(
            **best_parameters, objective="reg:absoluteerror", 
            tree_method="auto", random_state=seed, n_jobs=2))])

    predictions = walk_forward(xgboost_pipeline, year=2025)
    xgboost_seed_maes_2025.append(predictions["absolute_error"].mean())

results_2025 = [{"model": "Ridge", "mae": ridge_mae_2025}, {"model": "XGBoost (mean across seeds)", 
                "mae": sum(xgboost_seed_maes_2025) / len(xgboost_seed_maes_2025)}]


# creates the data structure for the baseline check. take only the data from the year 2025
evaluation_data = test_data.sort_values(["raceId", "driverId"])

# Starting-grid baseline
grid_scores = evaluation_data["gridPositionNumber"].fillna(float("inf"))            # again append the drivers with missing values at the end
grid_positions = grid_scores.groupby(evaluation_data["raceId"]).rank(method="first")        # rank drivers by their starting position within each race.

results_2025.append({"model": "Starting-grid baseline",
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], grid_positions)})

# Qualifying baseline
quali_scores = evaluation_data["qualificationPositionNumber"].fillna(float("inf"))
quali_positions = quali_scores.groupby(evaluation_data["raceId"]).rank(method="first")

results_2025.append({"model": "Qualifying baseline", 
                     "mae": mean_absolute_error(evaluation_data["positionDisplayOrder"], quali_positions)})

comparison_2025 = pd.DataFrame(results_2025).sort_values("mae")
display(comparison_2025.round(3))

,model,mae
0,Ridge,3.287
1,XGBoost (mean across seeds),3.296
3,Qualifying baseline,3.312
2,Starting-grid baseline,3.333
